# Chapter 08: FP8, KV Cache Quantization & Mixed Precision

**Goal:** Go beyond weight-only quantization — understand FP8 formats, KV cache quantization,
and mixed-precision strategies that make long-context inference possible.

Chapter 7 covered weight quantization (GPTQ, AWQ). This chapter covers:
- **FP8 formats** (E4M3, E5M2) — why floating-point quantization often beats integer
- **KV cache quantization** — a separate problem from weights (activations, not parameters)
- **Mixed precision** — different layers need different precision
- **Combined optimization** — INT4 weights + INT8 KV makes long-context feasible

**Model:** LLaMA-3.2-1B (d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers)

**Approach:** Fill in every `# YOUR CODE` section yourself — that's the learning.

**Hardware:** A100 80GB SXM4
- Measured HBM bandwidth: 1774 GB/s
- FP16 TFLOPS: 237
- Ridge point: ~134 FLOPS/byte

## 1. Beyond INT: Floating Point Quantization

**FP8** has two standard formats (IEEE-defined for Hopper GPUs):

| Format | Sign | Exponent | Mantissa | Range | Precision |
|--------|------|----------|----------|-------|-----------|
| FP16   | 1    | 5        | 10       | ~65K  | High      |
| FP8 E4M3 | 1 | 4        | 3        | ~240  | Medium    |
| FP8 E5M2 | 1 | 5        | 2        | ~57K  | Lower     |
| INT8   | -    | -        | -        | -128..127 | Uniform |

**Why FP8 > INT8 for many operations:**
- Neural network weights follow a bell curve (Gaussian-like), not uniform
- FP8 has more representable values near zero (where most weights live)
- INT8 wastes precision on large values that rarely occur

**E4M3 vs E5M2:**
- E4M3: more mantissa bits -> better precision, used for **weights and activations**
- E5M2: more exponent bits -> wider range, used for **gradients** (training)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda')
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

# Hardware constants
peak_bw = 1774   # GB/s
peak_tfl = 237    # TFLOPS FP16
ridge_point = peak_tfl * 1000 / peak_bw  # ~134 FLOPS/byte

# Model dimensions
d_model = 2048
n_heads = 32
n_kv_heads = 8
d_head = 64
d_ff = 8192
n_layers = 16

In [ ]:
# --- YOUR CODE: compare representable values of FP16, FP8-E4M3, INT8 ---

# FP16 representable values (sample)
fp16_values = []
for exp in range(-14, 16):  # FP16 exponent range
    for mant in range(1024):  # 10-bit mantissa
        val = (1 + mant / 1024) * (2 ** exp)
        if val < 10:  # Keep reasonable range for visualization
            fp16_values.append(val)
            fp16_values.append(-val)

# FP8 E4M3 representable values
# YOUR CODE: enumerate all FP8-E4M3 values
# Hint: 4-bit exponent (bias=7), 3-bit mantissa
fp8_e4m3_values = []
for exp in range(-6, 9):  # E4M3 exponent range (bias=7, special handling)
    for mant in range(8):  # 3-bit mantissa
        val = (1 + mant / 8) * (2 ** exp)  # Hint: (1 + mantissa/2^3) * 2^exp
        if val < 10:
            fp8_e4m3_values.append(val)
            fp8_e4m3_values.append(-val)

# INT8 representable values (uniform grid after scaling)
# YOUR CODE: with a typical scale mapping [-1, 1] range
int8_scale = 1.0 / 127
int8_values = [i * int8_scale for i in range(-127, 128)]  # Hint: uniform grid

# Plot density of representable values
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, values, name, color in [
    (axes[0], fp16_values, 'FP16 (5E + 10M)', '#1f77b4'),
    (axes[1], fp8_e4m3_values, 'FP8-E4M3 (4E + 3M)', '#ff7f0e'),
    (axes[2], int8_values, 'INT8 (uniform)', '#2ca02c'),
]:
    vals = [v for v in values if -2 < v < 2]
    ax.hist(vals, bins=200, color=color, alpha=0.7, density=True)
    ax.set_ylabel('Density')
    ax.set_title(f'{name}: {len(set(vals))} unique values in [-2, 2]')
    ax.grid(True, alpha=0.3)

axes[2].set_xlabel('Value')
fig.suptitle('Representable Values: FP formats cluster near zero (matching weight distributions)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"FP8-E4M3 has more values near zero -> better for Gaussian-distributed weights")
print(f"INT8 has uniform spacing -> wastes precision on rare large values")

## 2. Hopper's FP8 Tensor Cores

**H100 (Hopper)** has native FP8 tensor cores:
- FP8 matmul: **1979 TFLOPS** (2x FP16)
- FP16 matmul: **989 TFLOPS**

So FP8 gives both:
1. **2x less memory** (8-bit vs 16-bit) -> 2x less HBM traffic
2. **2x more compute** (native FP8 tensor cores)

On **A100** (no native FP8 tensor cores), we still get the memory benefit.
We can study FP8 via software emulation using PyTorch's `torch.float8_e4m3fn`.

In [ ]:
# --- YOUR CODE: calculate intensity at FP8, compare with FP16 on roofline ---

# Decode: y = Wx, W is (N, M), x is (M, 1)
M, N = 2048, 2048

formats = {
    'FP16':     {'bpw': 2.0, 'peak_tfl': 237},   # A100 FP16
    'FP8':      {'bpw': 1.0, 'peak_tfl': 237},   # A100: no native FP8, same compute as FP16
    'FP8 (H100)': {'bpw': 1.0, 'peak_tfl': 1979/1000 * 237},  # H100: 2x compute boost
    'INT4':     {'bpw': 0.5, 'peak_tfl': 237},   # A100: INT4 dequant + FP16 compute
}

print(f"Decode intensity analysis (batch=1, {M}x{N} layer):")
print(f"{'Format':<14} {'Bytes/W':>8} {'Intensity':>12} {'Theoretical':>15} {'vs FP16':>10}")
print("-" * 65)

fp16_tok_s = None
for fmt, props in formats.items():
    flops = 2 * M * N
    weight_bytes = M * N * props['bpw']
    total_bytes = weight_bytes + M * 2 + N * 2  # +input +output in FP16
    intensity = flops / total_bytes  # YOUR CODE

    # Theoretical tok/s (memory-bound: limited by bandwidth)
    time_s = total_bytes / (peak_bw * 1e9)
    tok_s = 1.0 / time_s
    if fp16_tok_s is None:
        fp16_tok_s = tok_s

    print(f"{fmt:<14} {props['bpw']:>8.1f} {intensity:>10.1f} F/B {tok_s:>13,.0f} t/s {tok_s/fp16_tok_s:>8.1f}x")

print(f"\nOn A100: FP8 gives ~2x bandwidth benefit (same as INT8).")
print(f"On H100: FP8 gives 2x bandwidth + 2x compute = potentially 4x for compute-bound cases.")

## 3. Mixed Precision Strategies

Not everything should be quantized the same way.

| Component | Recommended Precision | Why |
|-----------|---------------------|-----|
| FFN weights | INT4/FP8 | Large matrices, less sensitive |
| Attention Q/K/V/O weights | INT4/FP8 | Moderately sensitive |
| Attention scores (QK^T) | FP32 | Softmax needs precision for numerical stability |
| RMSNorm | FP32 | Small tensors, rsqrt needs precision |
| Activations | FP16 | Intermediate values, moderate sensitivity |
| KV cache | INT8/FP8 | Can tolerate some error (see Section 4) |

**Key principle:** quantize what's big (weights, KV cache), keep precise what's small but sensitive (norms, softmax).

In [ ]:
# --- YOUR CODE: map each LLaMA layer to its optimal precision and calculate total memory ---

# LLaMA-3.2-1B layer components (per transformer block)
layer_components = [
    # (name, param_count, recommended_precision, bytes_per_param)
    ('q_proj weight',    d_model * d_model,       'INT4',  0.5),
    ('k_proj weight',    d_model * (n_kv_heads * d_head), 'INT4', 0.5),
    ('v_proj weight',    d_model * (n_kv_heads * d_head), 'INT4', 0.5),
    ('o_proj weight',    d_model * d_model,       'INT4',  0.5),
    ('gate_proj weight', d_model * d_ff,          'INT4',  0.5),
    ('up_proj weight',   d_model * d_ff,          'INT4',  0.5),
    ('down_proj weight', d_ff * d_model,          'INT4',  0.5),
    ('input_layernorm',  d_model,                 'FP32',  4.0),
    ('post_attn_norm',   d_model,                 'FP32',  4.0),
]

# Non-block components
embedding_params = 128256 * d_model  # vocab_size * d_model
lm_head_params = 128256 * d_model
final_norm_params = d_model

# YOUR CODE: calculate total memory for mixed precision
total_block_bytes_fp16 = 0
total_block_bytes_mixed = 0

print(f"Per-block memory (mixed precision):")
print(f"{'Component':<25} {'Params':>12} {'FP16 (MB)':>10} {'Mixed (MB)':>10} {'Precision':>10}")
print("-" * 72)

for name, params, precision, bpp in layer_components:
    fp16_bytes = params * 2
    mixed_bytes = params * bpp
    total_block_bytes_fp16 += fp16_bytes
    total_block_bytes_mixed += mixed_bytes
    print(f"{name:<25} {params:>12,} {fp16_bytes/1e6:>10.2f} {mixed_bytes/1e6:>10.2f} {precision:>10}")

print(f"{'--- Block total ---':<25} {'':>12} {total_block_bytes_fp16/1e6:>10.2f} {total_block_bytes_mixed/1e6:>10.2f}")

# Full model
model_fp16 = (total_block_bytes_fp16 * n_layers + (embedding_params + lm_head_params) * 2 + final_norm_params * 4)
model_mixed = (total_block_bytes_mixed * n_layers + (embedding_params + lm_head_params) * 0.5 + final_norm_params * 4)

print(f"\nFull model ({n_layers} layers + embeddings):")
print(f"  FP16:  {model_fp16/1e9:.2f} GB")
print(f"  Mixed: {model_mixed/1e9:.2f} GB")
print(f"  Savings: {(model_fp16 - model_mixed)/1e9:.2f} GB ({model_fp16/model_mixed:.1f}x smaller)")

## 4. KV Cache Quantization

**This is a separate problem from weight quantization.**

- **Weights** are fixed after training — we can carefully calibrate quantization offline
- **KV cache** stores activations that change every forward pass — needs dynamic quantization

The KV cache is also more **sensitive** to quantization error because:
- Attention scores are a softmax over `Q @ K^T / sqrt(d_head)` — small K errors get amplified
- V errors directly corrupt the attention output

But the KV cache can be **huge** for long sequences, so quantizing it is essential.

In [ ]:
# --- YOUR CODE: calculate KV cache at FP16, FP8, INT4 for various models ---

def kv_cache_size(n_layers, n_kv_heads, d_head, seq_len, bytes_per_element):
    """Calculate KV cache size.
    KV cache = 2 (K and V) * n_layers * n_kv_heads * d_head * seq_len * bytes_per_element
    """
    # YOUR CODE
    return 2 * n_layers * n_kv_heads * d_head * seq_len * bytes_per_element  # Hint: 2 for K and V

# LLaMA-3.2-1B
print("LLaMA-3.2-1B KV Cache (8 KV heads, d_head=64, 16 layers):")
print(f"{'Seq len':>10} {'FP16 (MB)':>12} {'FP8 (MB)':>12} {'INT4 (MB)':>12}")
print("-" * 50)
for seq in [512, 2048, 4096, 8192, 32768, 131072]:
    fp16 = kv_cache_size(16, 8, 64, seq, 2.0) / 1e6
    fp8 = kv_cache_size(16, 8, 64, seq, 1.0) / 1e6
    int4 = kv_cache_size(16, 8, 64, seq, 0.5) / 1e6
    print(f"{seq:>10,} {fp16:>12.1f} {fp8:>12.1f} {int4:>12.1f}")

# LLaMA-70B for comparison
print(f"\nLLaMA-70B KV Cache (8 KV heads, d_head=128, 80 layers):")
print(f"{'Seq len':>10} {'FP16 (GB)':>12} {'FP8 (GB)':>12} {'INT4 (GB)':>12}")
print("-" * 50)
for seq in [4096, 8192, 32768, 131072]:
    fp16 = kv_cache_size(80, 8, 128, seq, 2.0) / 1e9
    fp8 = kv_cache_size(80, 8, 128, seq, 1.0) / 1e9
    int4 = kv_cache_size(80, 8, 128, seq, 0.5) / 1e9
    print(f"{seq:>10,} {fp16:>12.2f} {fp8:>12.2f} {int4:>12.2f}")

print(f"\nAt 131K tokens, LLaMA-70B KV cache in FP16 = {kv_cache_size(80, 8, 128, 131072, 2.0)/1e9:.1f} GB")
print(f"That's more than an entire A100's 80 GB VRAM — just for the KV cache!")
print(f"INT4 KV reduces this to {kv_cache_size(80, 8, 128, 131072, 0.5)/1e9:.1f} GB — feasible.")

## 5. Per-Token vs Per-Channel Calibration

KV values change every forward pass, so we need **dynamic quantization** (compute scale on-the-fly).

Two strategies:
- **Per-tensor:** one scale for the entire K (or V) tensor -> fast but inaccurate
- **Per-token:** one scale per token position -> better accuracy, more overhead

Per-token is worth it because different tokens have very different K/V magnitudes.

In [ ]:
# --- YOUR CODE: implement per-token quantization for K and V tensors ---

def quantize_per_tensor(x, bits=8):
    """Single scale for entire tensor."""
    # YOUR CODE
    qmax = 2 ** (bits - 1) - 1
    scale = x.abs().max() / qmax  # Hint: one global scale
    x_q = torch.clamp(torch.round(x / scale), -qmax - 1, qmax).to(torch.int8)
    return x_q, scale

def dequantize_per_tensor(x_q, scale):
    return x_q.float() * scale

def quantize_per_token(x, bits=8):
    """Separate scale per token (last dim stays, quantize over head_dim)."""
    # x shape: (batch, n_heads, seq_len, d_head)
    # YOUR CODE: compute scale per token
    qmax = 2 ** (bits - 1) - 1
    # Scale shape: (batch, n_heads, seq_len, 1)
    scale = x.abs().amax(dim=-1, keepdim=True) / qmax  # Hint: per-token scale
    scale = scale.clamp(min=1e-8)
    x_q = torch.clamp(torch.round(x / scale), -qmax - 1, qmax).to(torch.int8)
    return x_q, scale

def dequantize_per_token(x_q, scale):
    return x_q.float() * scale

# Simulate K tensor from attention (batch=1, 8 KV heads, seq=512, d_head=64)
torch.manual_seed(42)
K = torch.randn(1, 8, 512, 64, dtype=torch.float32, device=device)
# Make some tokens have much larger magnitudes (realistic: some positions are salient)
K[:, :, 0:10, :] *= 5.0   # First few tokens (BOS, system prompt) are often high-magnitude
K[:, :, 100:105, :] *= 3.0

# --- YOUR CODE: compare per-tensor vs per-token quantization ---
K_q_tensor, scale_tensor = quantize_per_tensor(K, bits=8)
K_deq_tensor = dequantize_per_tensor(K_q_tensor, scale_tensor)

K_q_token, scale_token = quantize_per_token(K, bits=8)
K_deq_token = dequantize_per_token(K_q_token, scale_token)

# Measure error
mse_tensor = ((K - K_deq_tensor) ** 2).mean().item()
mse_token = ((K - K_deq_token) ** 2).mean().item()
max_err_tensor = (K - K_deq_tensor).abs().max().item()
max_err_token = (K - K_deq_token).abs().max().item()

print(f"K tensor quantization (INT8):")
print(f"{'Method':<20} {'MSE':>12} {'Max error':>12} {'Scale overhead':>15}")
print("-" * 62)
print(f"{'Per-tensor':<20} {mse_tensor:>12.8f} {max_err_tensor:>12.6f} {'1 float':>15}")
print(f"{'Per-token':<20} {mse_token:>12.8f} {max_err_token:>12.6f} {f'{512} floats':>15}")
print(f"{'Improvement':<20} {mse_tensor/mse_token:>12.1f}x {max_err_tensor/max_err_token:>12.1f}x")
print(f"\nPer-token is {mse_tensor/mse_token:.0f}x better MSE with minimal overhead.")

## 6. Implementing KV Cache Quantization

In a real LLaMA forward pass, we quantize K and V right after computing them,
then dequantize just before the attention computation.

```
K = k_proj(hidden_states)   # FP16
V = v_proj(hidden_states)   # FP16
K_q, K_scale = quantize(K)  # INT8 + scale (stored in cache)
V_q, V_scale = quantize(V)  # INT8 + scale (stored in cache)
...
K = dequantize(K_q, K_scale)  # Back to FP16 for attention
V = dequantize(V_q, V_scale)
attn_output = attention(Q, K, V)
```

In [ ]:
# --- YOUR CODE: quantize KV cache and measure attention output error ---

# Simulate a full attention layer
batch, seq_len = 1, 512

# Generate Q, K, V (realistic scale)
torch.manual_seed(42)
Q = torch.randn(batch, n_heads, seq_len, d_head, dtype=torch.float16, device=device) * 0.1
K = torch.randn(batch, n_kv_heads, seq_len, d_head, dtype=torch.float16, device=device) * 0.1
V = torch.randn(batch, n_kv_heads, seq_len, d_head, dtype=torch.float16, device=device) * 0.1

# GQA: expand KV heads to match Q heads (32 Q heads / 8 KV heads = 4x repeat)
def expand_kv(kv, n_rep):
    """Repeat KV heads for GQA."""
    batch, n_kv, seq, dim = kv.shape
    return kv[:, :, None, :, :].expand(batch, n_kv, n_rep, seq, dim).reshape(batch, n_kv * n_rep, seq, dim)

n_rep = n_heads // n_kv_heads  # 4

def attention_output(Q, K, V):
    """Compute attention output."""
    K_expanded = expand_kv(K, n_rep)
    V_expanded = expand_kv(V, n_rep)
    scores = torch.matmul(Q, K_expanded.transpose(-2, -1)) / (d_head ** 0.5)
    attn_weights = torch.softmax(scores.float(), dim=-1).half()
    return torch.matmul(attn_weights, V_expanded)

# Baseline: FP16 KV
output_fp16 = attention_output(Q, K, V)

# --- YOUR CODE: INT8 KV cache ---
K_q8, K_s8 = quantize_per_token(K.float(), bits=8)  # YOUR CODE
V_q8, V_s8 = quantize_per_token(V.float(), bits=8)  # YOUR CODE
K_deq8 = dequantize_per_token(K_q8, K_s8).half()
V_deq8 = dequantize_per_token(V_q8, V_s8).half()
output_int8 = attention_output(Q, K_deq8, V_deq8)

# --- YOUR CODE: FP8 KV cache (simulated via clamped FP16) ---
# Simulate FP8 by rounding to nearest FP8-representable value
if hasattr(torch, 'float8_e4m3fn'):
    K_fp8 = K.to(torch.float8_e4m3fn).to(torch.float16)
    V_fp8 = V.to(torch.float8_e4m3fn).to(torch.float16)
else:
    # Fallback: simulate FP8 precision (3-bit mantissa = 8 levels per exponent)
    def simulate_fp8(x):
        sign = x.sign()
        abs_x = x.abs().clamp(min=1e-7)
        exp = torch.floor(torch.log2(abs_x))
        mant = abs_x / (2.0 ** exp)
        mant_q = torch.round(mant * 8) / 8  # 3-bit mantissa
        return sign * mant_q * (2.0 ** exp)
    K_fp8 = simulate_fp8(K)
    V_fp8 = simulate_fp8(V)
output_fp8 = attention_output(Q, K_fp8, V_fp8)

# Compare
def relative_error(output, baseline):
    return ((output - baseline) ** 2).mean().sqrt() / (baseline ** 2).mean().sqrt()

print(f"Attention output error with quantized KV cache:")
print(f"{'KV Format':<15} {'Relative Error':>15} {'Max Abs Error':>15} {'Memory (per KV)':>15}")
print("-" * 65)
kv_elements = 2 * n_kv_heads * seq_len * d_head
print(f"{'FP16 (baseline)':<15} {'0':>15} {'0':>15} {kv_elements*2/1e6:>13.2f} MB")
print(f"{'INT8 (per-tok)':<15} {relative_error(output_int8, output_fp16).item():>15.6f} {(output_int8-output_fp16).abs().max().item():>15.6f} {kv_elements*1/1e6:>13.2f} MB")
print(f"{'FP8-E4M3':<15} {relative_error(output_fp8, output_fp16).item():>15.6f} {(output_fp8-output_fp16).abs().max().item():>15.6f} {kv_elements*1/1e6:>13.2f} MB")

## 7. Combined Optimization: Quantized Weights + Quantized KV

The real power is combining weight quantization AND KV cache quantization.

**LLaMA-70B at seq=32K — the memory budget problem:**

| Component | FP16 | INT4 weights + INT8 KV |
|-----------|------|------------------------|
| Weights | 140 GB | 35 GB |
| KV cache | 10 GB | 5 GB |
| Total | 150 GB | 40 GB |
| Fits on... | 2x A100 | 1x A100 |

This is what makes long-context inference on a single GPU possible.

In [ ]:
# --- YOUR CODE: INT4 weights + INT8 KV cache memory analysis ---

def total_memory_gb(model_name, n_layers, d_model, n_kv_heads, d_head, d_ff,
                     vocab_size, seq_len, weight_bpw, kv_bpe):
    """Calculate total inference memory."""
    # Weight memory
    # Per block: q_proj + k_proj + v_proj + o_proj + gate + up + down
    n_heads_q = d_model // d_head
    kv_dim = n_kv_heads * d_head
    params_per_block = (
        d_model * d_model +      # q_proj
        d_model * kv_dim +        # k_proj
        d_model * kv_dim +        # v_proj
        d_model * d_model +       # o_proj
        d_model * d_ff +          # gate_proj
        d_model * d_ff +          # up_proj
        d_ff * d_model            # down_proj
    )
    weight_bytes = (params_per_block * n_layers + vocab_size * d_model * 2) * weight_bpw

    # KV cache memory
    # YOUR CODE: 2 * n_layers * n_kv_heads * d_head * seq_len * kv_bpe
    kv_bytes = 2 * n_layers * n_kv_heads * d_head * seq_len * kv_bpe

    return weight_bytes / 1e9, kv_bytes / 1e9

# LLaMA-70B analysis
configs = [
    ('FP16 weights + FP16 KV',     2.0, 2.0),
    ('INT8 weights + FP16 KV',     1.0, 2.0),
    ('INT4 weights + FP16 KV',     0.5, 2.0),
    ('INT4 weights + INT8 KV',     0.5, 1.0),
    ('INT4 weights + FP8 KV',      0.5, 1.0),
    ('INT4 weights + INT4 KV',     0.5, 0.5),
]

seq_len = 32768
print(f"LLaMA-70B memory at seq_len={seq_len:,}:")
print(f"{'Config':<32} {'Weights':>10} {'KV Cache':>10} {'Total':>10} {'Fits on':>15}")
print("-" * 82)

for name, w_bpw, kv_bpe in configs:
    w_gb, kv_gb = total_memory_gb(
        'LLaMA-70B', n_layers=80, d_model=8192, n_kv_heads=8, d_head=128,
        d_ff=28672, vocab_size=128256, seq_len=seq_len,
        weight_bpw=w_bpw, kv_bpe=kv_bpe
    )
    total = w_gb + kv_gb
    if total <= 24:
        hw = '1x RTX 4090'
    elif total <= 48:
        hw = '1x A6000'
    elif total <= 80:
        hw = '1x A100-80GB'
    elif total <= 160:
        hw = '2x A100-80GB'
    else:
        hw = '4+ A100s'
    print(f"{name:<32} {w_gb:>8.1f} GB {kv_gb:>8.1f} GB {total:>8.1f} GB {hw:>15}")

print(f"\nINT4 weights + INT8 KV is the sweet spot:")
print(f"  - 4x weight compression with ~0.5 perplexity loss")
print(f"  - 2x KV compression with minimal attention error")
print(f"  - Makes 70B at 32K context fit on a single A100")

## 8. Quality Benchmarks

How much quality do we lose with each quantization combination?

We create a matrix: {FP16, INT8, INT4} weights x {FP16, INT8, FP8} KV cache.

Real perplexity measurement requires running the full model on a calibration dataset.
Here we approximate by measuring reconstruction error at the layer level.

In [ ]:
# --- YOUR CODE: measure error impact of each quantization combination ---

# Simulate a single attention + FFN block with different quantization
torch.manual_seed(42)

batch, seq = 1, 512
x = torch.randn(batch, seq, d_model, dtype=torch.float16, device=device) * 0.1

# Create weight matrices (simulated)
W_q = torch.randn(d_model, d_model, dtype=torch.float16, device=device) * 0.02
W_k = torch.randn(n_kv_heads * d_head, d_model, dtype=torch.float16, device=device) * 0.02
W_v = torch.randn(n_kv_heads * d_head, d_model, dtype=torch.float16, device=device) * 0.02

def quantize_weight(W, bits):
    if bits == 16:
        return W  # No quantization
    qmax = 2 ** (bits - 1) - 1
    scale = W.float().abs().max() / qmax
    W_q = torch.clamp(torch.round(W.float() / scale), -qmax - 1, qmax)
    return (W_q * scale).half()

def quantize_kv(tensor, fmt):
    if fmt == 'FP16':
        return tensor
    elif fmt == 'INT8':
        t_q, t_s = quantize_per_token(tensor.float(), bits=8)
        return dequantize_per_token(t_q, t_s).half()
    elif fmt == 'FP8':
        if hasattr(torch, 'float8_e4m3fn'):
            return tensor.to(torch.float8_e4m3fn).to(torch.float16)
        else:
            sign = tensor.sign()
            abs_x = tensor.abs().clamp(min=1e-7)
            exp = torch.floor(torch.log2(abs_x.float()))
            mant = abs_x.float() / (2.0 ** exp)
            mant_q = torch.round(mant * 8) / 8
            return (sign * mant_q * (2.0 ** exp)).half()

# Baseline: full FP16
Q_fp16 = x @ W_q.T
K_fp16 = x @ W_k.T
V_fp16 = x @ W_v.T
K_fp16 = K_fp16.view(batch, seq, n_kv_heads, d_head).transpose(1, 2)
V_fp16 = V_fp16.view(batch, seq, n_kv_heads, d_head).transpose(1, 2)
Q_base = Q_fp16.view(batch, seq, n_heads, d_head).transpose(1, 2)
output_baseline = attention_output(Q_base, K_fp16, V_fp16)

# --- YOUR CODE: create matrix of weight x KV quantization combinations ---
weight_configs = [('FP16', 16), ('INT8', 8), ('INT4', 4)]
kv_configs = ['FP16', 'INT8', 'FP8']

print(f"Attention output error matrix (relative to FP16 baseline):")
print(f"{'':>16}", end='')
for kv_fmt in kv_configs:
    print(f"{f'KV={kv_fmt}':>16}", end='')
print()
print("-" * (16 + 16 * len(kv_configs)))

for w_name, w_bits in weight_configs:
    print(f"{'W=' + w_name:>16}", end='')
    for kv_fmt in kv_configs:
        # Quantize weights
        W_q_q = quantize_weight(W_q, w_bits)
        W_k_q = quantize_weight(W_k, w_bits)
        W_v_q = quantize_weight(W_v, w_bits)

        # Compute Q, K, V with quantized weights
        Q_out = x @ W_q_q.T
        K_out = x @ W_k_q.T
        V_out = x @ W_v_q.T

        K_out = K_out.view(batch, seq, n_kv_heads, d_head).transpose(1, 2)
        V_out = V_out.view(batch, seq, n_kv_heads, d_head).transpose(1, 2)
        Q_out = Q_out.view(batch, seq, n_heads, d_head).transpose(1, 2)

        # Quantize KV cache
        K_final = quantize_kv(K_out, kv_fmt)
        V_final = quantize_kv(V_out, kv_fmt)

        # Compute attention
        output = attention_output(Q_out, K_final, V_final)
        rel_err = relative_error(output, output_baseline).item()
        print(f"{rel_err:>16.6f}", end='')
    print()

print(f"\nRows = weight quantization, Columns = KV cache quantization")
print(f"Lower = better. Weight quantization dominates the error budget.")

## Revision Notes

*Fill this in after running all sections.*

---

**FP8 E4M3 vs E5M2:**
- E4M3 is for: ___
- E5M2 is for: ___

**Why FP8 > INT8 for neural nets:**



**KV cache sizes (LLaMA-70B, seq=32K):**
- FP16: ___ GB
- INT8: ___ GB
- INT4: ___ GB

**Per-token vs per-tensor KV quantization:**
- MSE improvement: ___x

**Best combined config for LLaMA-70B at 32K context:**



**Quality impact matrix (most important finding):**



**What makes long-context inference possible:**



**What surprised me:**



---
*Next: Chapter 9 — Batching & Throughput Optimization*